<a href="https://colab.research.google.com/github/suhacai12-star/mnist-numpy-mlp/blob/main/implement-MLP-numpy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab 시작 페이지

In [3]:
#데이터 전처리
# 1차원으로 펼치기 , 부동소수점으로 고치기

x_train = x_train.reshape(60000,-1)
x_test = x_test.reshape(10000,-1)
# one hot encoding
y_etrain = np.zeros((60000,10))
for i in range(len(y_train)):
  y_etrain[i][y_train[i]] = 1
y_etest = np.zeros((10000,10))
for i in range(len(y_test)):
  y_etest[i][y_test[i]] = 1


x_train = x_train.astype('float64')  # 225,0
x_test = x_test.astype('float64') #225.0
print(y_train[0])
print(y_etrain[0])


5
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]


In [ ]:
# 이미지와 정답 확인하기
import matplotlib.pyplot as plt

index = 1

plt.imshow(x_train[index].reshape(28,28), cmap = "gray")
plt.title(f"Label: {y_train[index]}")
plt.axis("off") # 축 눈금 안보이게 하기
plt.show()

In [15]:
import time
class MLP:

  def __init__(self,lr,x_train,y_train,x_test,y_test,batch):
    self.lr = lr
    self.x_train = x_train
    self.y_train  =y_train
    self.x_test  = x_test
    self.y_test = y_test
    # input (input, batch)
    # w*x+b 형태

    self.B_x = np.zeros((784,100))
    self.B_y = np.zeros((10,100))
    self.B = batch
    # 가중치
    self.w1 = np.random.normal(0,np.sqrt(2/(784+512)),size =(512,784)) # xaiver (output,input)
    self.w2 = np.random.normal(0,np.sqrt(2/512),size = (128,512))# he
    self.w3 = np.random.normal(0,np.sqrt(2/((128+10))),size =(10,128))# xaiver
    self.b1 = np.zeros((512,1))
    self.b2 = np.zeros((128,1))
    self.b3 = np.zeros((10,1))

    self.z1 = np.zeros((512,self.B))
    self.a1 = np.zeros((512,self.B))
    self.z2 = np.zeros((128,self.B))
    self.a2 = np.zeros((128,self.B))
    self.z3 = np.zeros((10,self.B))
    self.predict = np.zeros((10,self.B))



  def sigmoid(self,x):
    return 1/(1+np.exp(-x))
  def Desigmoid(self,o): # output기준
    return o*(1-o)
  def Relu(self,o):
    return np.maximum(0,o)
  def DeRelu(self,x):  # 입력값 기준
   return (x>0).astype('float64')
  def softmax(self,x):
    c = np.max(x,axis = 0,keepdims =True)
    exp_x = np.exp(x-c)

    return exp_x/np.sum(exp_x, axis = 0, keepdims = True) # 열벡터 (10,batch) -> (1,batch)

  def mean(self,x): # x: (output,B)
    B = x.shape[1]
    tmp = np.sum(x,axis = 1,keepdims = True)/B # (output,1)
    return tmp



  def CN(self,p):
    B = len(self.B_y[0])
    total = -np.sum(self.B_y*np.log(p),axis = 0,keepdims = True)
    avg = np.sum(total)/B

    return avg
  def outputdelta(self,o): #(10,B)
    return o-self.B_y



  def forward(self): # (784,B)
    self.z1 = self.w1 @ self.B_x + self.b1 # (256,B)
    self.a1 = self.sigmoid(self.z1) # (256,B)
    self.z2 = self.w2 @ self.a1 + self.b2 # (128,B)
    self.a2 = self.Relu(self.z2) # (128,B)
    self.z3 = self.w3 @ self.a2 + self.b3 # (10,B)
    self.predict = self.softmax(self.z3) # (10,B)
  # (10,B)
    return self.predict

  def Loss(self): # (10,B)
    result = self.CN(self.predict) # (1,1)
    return result
  def back(self):
    outputdelta = self.outputdelta(self.predict) # (10,B) (B,128)
    self.b3 -= self.lr * self.mean(outputdelta) # (10,1)

    hiddenoutputdelta2 = (self.w3.T @ outputdelta) * self.DeRelu(self.a2)
    self.w3 -= self.lr * (outputdelta @ self.a2.T)/self.B
          #(128,B)

    self.b2 -= self.lr * self.mean(hiddenoutputdelta2) # (128,1)
    hiddenoutputdelta1 = (self.w2.T @ hiddenoutputdelta2) * self.Desigmoid(self.a1) # (256,B)
    self.w2 -= self.lr * (hiddenoutputdelta2 @ self.a1.T)/self.B # (128,256)

    self.b1 -= self.lr * self.mean(hiddenoutputdelta1) # (256,1)
    self.w1 -= self.lr * (hiddenoutputdelta1 @ self.B_x.T)/self.B # (256,784)

  def training(self):

    for epoch in range(100):

      total_loss = 0.0
      batch_count = 0

      for batch in range(0,len(self.x_train),self.B):
        self.B_x = self.x_train[batch:batch+self.B].T # 0~99, 100~199
        self.B_y = self.y_train[batch:batch+self.B].T

        start = time.perf_counter()

        output = self.forward()
        forward_time = time.perf_counter() - start
        start = time.perf_counter()
        loss = self.Loss()
        loss_time = time.perf_counter() - start

        start = time.perf_counter()

        self.back()
        total_loss += loss
        batch_count += 1
        back_time = time.perf_counter() - start

        if batch == 0:
                print(f"Forward: {forward_time:.6f}초")
                print(f"Loss:    {loss_time:.6f}초")
                print(f"Back:    {back_time:.6f}초")
      print(f"epoch: {epoch}, loss: {total_loss / batch_count}")

  def evaluate(self):
    count = 0
    for i in range(0,10000,100):
      self.B_x = self.x_test[i:i+100].T
      self.B_y = self.y_test[i:i+100].T
      self.forward()
      ans = np.argmax(self.predict, axis=0)
      target = np.argmax(self.B_y, axis=0)
      for j in range(100):

        if ans[j] == target[j]:

           count +=1
        else:

          print("B_x shape:", self.B_x.shape)
          print("j:", j)
          print(f"predict: {ans[j]}")
          img = self.B_x.T[j].reshape(28, 28)

          plt.imshow(img, cmap="gray")
          plt.title(f"Label: {np.argmax(self.B_y.T[j])}")
          plt.axis("off")
          plt.show()
    print(f"accuracy:  {(count/10000)*100}")















In [ ]:
model = MLP(0.01,x_train,y_etrain,x_test,y_etest,100)
model.training()
model.evaluate()

In [78]:
model.evaluate()

/tmp/ipykernel_3751/1520288622.py:34: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-x))


accuracy:  91.61
